In [0]:
%sql
--This Databricks SQL statement creates or refreshes a materialized view named 'workspace.gold.stock_daily_returns'.
--1-The daily percentage returns for each stock ticker.
--For each row, it computes the percent change in 'close' price compared to the previous day's 'close' for the same ticker.
--The calculation uses the LAG window function to access the prior day's close price, partitions by ticker, and orders by date.
--The result is rounded to 4 decimal places and stored as 'daily_return_pct'.
--The source data is from the 'stocks_bronze' table.

CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.stock_daily_returns AS
SELECT
    ticker,
    date,
    close,
    ROUND(
        (close - LAG(close) OVER (PARTITION BY ticker ORDER BY date)) /
        LAG(close) OVER (PARTITION BY ticker ORDER BY date) * 100,
        4
    ) AS daily_return_pct
FROM stocks_bronze;

select* from workspace.gold.stock_daily_returns;

--2. Moving Averages (trend analysis)
CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.stocks_moving_averages AS
SELECT
    ticker,
    date,
    close,
    AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)  AS ma_7,
    AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS ma_30
FROM stocks_bronze;

select* from workspace.gold.stocks_moving_averages;

-- 3- 3. Volatility (risk measure)
CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.stocks_volatility AS
SELECT
    ticker,
    DATE_TRUNC('month', date) AS month,
    ROUND(STDDEV(close), 2)   AS price_volatility
FROM stocks_bronze
GROUP BY ticker, DATE_TRUNC('month', date);